# Title


**IMDB Movie Review Sentiment Analysis**

# Problem Statement



Nowadays, people share thousands of movie reviews online. These reviews contain different opinions, such as positive and negative feedback.

Manually reading and identifying whether a review is positive or negative takes a lot of time.

Therefore, the problem is to develop a Machine Learning model that automatically analyzes movie reviews and classifies them as Positive or Negative based on the text.

# Main Objective

1. Collect movie reviews from the IMDB dataset.

2. Clean and preprocess the review text.

3. Convert text into numerical features using Fast Text.

4. Train different ANN Architectures.

5. Compare their performance using F1 Score.

7. Predict whether a new movie review is Positive or Negative.

# Dataset Loading and Data Overivew

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install optuna gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 132.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [optuna]


In [3]:
import pandas as pd
import numpy as np
import re
import html
import nltk
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.models import FastText
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
                                        Dense,
                                        Dropout,
                                        BatchNormalization
                                    )

from sklearn.metrics import accuracy_score, classification_report

# Download required resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [4]:
df = pd.read_csv('/content/drive/MyDrive/ML/IMDB Dataset.csv')

In [5]:
# Display first 5 rows
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:


# Check number of rows and columns
print("Shape:", df.shape)

Shape: (50000, 2)


In [7]:
# Check column names
print("Columns:", df.columns.tolist())

Columns: ['review', 'sentiment']


In [8]:
# Check data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


#**Data cleaning**

In [9]:
df.isnull().sum()

,0
review,0
sentiment,0


**Observation**

There is no missing content


**Check duplicate rows**

In [10]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 418


In [11]:


df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

**Observation**

There are totla 418 duplicate values are available so then i remove that values

In [12]:
print(df['sentiment'].value_counts())
print(df['sentiment'].unique())

sentiment
positive    24884
negative    24698
Name: count, dtype: int64
['positive' 'negative']


In [13]:
stop_words = set(stopwords.words('english'))

# Preserve important negation words
stop_words -= {'no', 'nor', 'not'}

lemmatizer = WordNetLemmatizer()


def preprocess_text(text):

    # 1. Handle missing values
    if pd.isna(text):
        return ""

    text = str(text)

    # 2. Decode HTML entities
    text = html.unescape(text)

    # 3. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # 4. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # 5. Convert to lowercase
    text = text.lower()

    # 6. Expand common contractions
    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"\bcan't\b", "cannot", text)
    text = re.sub(r"\bshan't\b", "shall not", text)
    text = re.sub(r"\bain't\b", "is not", text)
    text = re.sub(r"n't\b", " not", text)
    text = re.sub(r"'re\b", " are", text)
    text = re.sub(r"'ve\b", " have", text)
    text = re.sub(r"'ll\b", " will", text)
    text = re.sub(r"'m\b", " am", text)
    text = re.sub(r"'d\b", " would", text)

    # 7. Remove punctuation, digits and special characters
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 8. Remove extra spaces and tokenize
    tokens = text.split()

    # 9. Remove stopwords while preserving negations
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    # 10. Lemmatization
    tokens = [
                lemmatizer.lemmatize(word, pos='v')
                for word in tokens
             ]

    # 11. Return processed text
    return tokens

In [14]:
df['processed_review'] = df['review'].apply(preprocess_text)

In [15]:
df[['review', 'processed_review', 'sentiment']].head()

,review,processed_review,sentiment
0,One of the other reviewers has mentioned that ...,"[one, reviewers, mention, watch, oz, episode, ...",positive
1,A wonderful little production. <br /><br />The...,"[wonderful, little, production, film, techniqu...",positive
2,I thought this was a wonderful way to spend ti...,"[think, wonderful, way, spend, time, hot, summ...",positive
3,Basically there's a family where a little boy ...,"[basically, family, little, boy, jake, think, ...",negative
4,"Petter Mattei's ""Love in the Time of Money"" is...","[petter, mattei, love, time, money, visually, ...",positive


# Fast Text


In [16]:
corpus = df['processed_review'].tolist()

In [17]:
Fast_text_model = FastText( sentences=corpus,
                            vector_size=100,
                            window=5,
                            min_count=1,
                            epochs=20
                          )

In [18]:
# Function to convert a review into an average word vector
def review_to_vector(review, model,vector_size):
  vectors = [model.wv[word] for word in review if word in model.wv]
  if vectors:
    return np.mean(vectors, axis=0)
  else:
    return np.zeros(vector_size)

vector_size = 100
review_vectors = np.array([review_to_vector(review, Fast_text_model, vector_size) for review in corpus])
print(review_vectors)
print(review_vectors.shape)


[[ 0.47494093  0.7111176  -0.51332456 ...  0.27543616 -0.02289332
   0.12283625]
 [ 0.20630793  0.68720126  0.2173616  ...  0.3960992   0.15999283
  -0.0995836 ]
 [ 0.35231966  0.26422554 -0.589785   ...  0.31521544  0.41463318
  -0.2990615 ]
 ...
 [ 0.79123825  0.67679554 -0.6855804  ...  0.09429734 -0.10406285
   0.02260836]
 [ 0.65120244  0.9121138   0.25523093 ... -0.4706334   0.01994912
   0.00536721]
 [-0.5276815   0.4450725  -1.0645225  ...  0.69733614 -0.14401202
  -0.22104883]]
(49582, 100)


#**Separate features and target**

In [19]:
X = review_vectors
y = df['sentiment']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (49582, 100)
Target shape: (49582,)


#**Encode sentiment labels**

In [20]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

print(le.classes_)

['negative' 'positive']


**Observation:**

Convert positive and negative labels into numerical values.

#**Split the dataset**

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
                                                    X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y
)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 39665
Testing samples: 9917


In [22]:
X_train = pd.DataFrame(X_train)
X_test = pd.DataFrame(X_test)
y_train = pd.DataFrame(y_train)
y_test = pd.DataFrame(y_test)

**Observation:**

We'll use 80% for training ,20% for testing, preserving the sentiment proportions with stratify = y

# ANN Architectures

# Defining metrics


In [23]:
import keras
f1_metric = keras.metrics.F1Score(threshold=0.5, name="f1_score")

#**Architecture 1**

- Input layer
- One hidden layer (16 neurons)
- Output layer (1 neuron)

In [24]:
# Shallow Network
model_1 = Sequential([
                      Dense(16, activation='relu', input_shape = (X_train.shape[1],)),
                      Dense(1, activation='sigmoid')
                     ])
# Compile
model_1.compile(
                  optimizer = 'adam',
                  loss = 'binary_crossentropy',
                  metrics = [keras.metrics.F1Score(name='f1_score')]

)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


**Model Summary**

In [25]:
model_1.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │         1,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,633 (6.38 KB)

 Trainable params: 1,633 (6.38 KB)

 Non-trainable params: 0 (0.00 B)

**Train the Model**

In [26]:
history = model_1.fit(
                       X_train, y_train,
                       validation_split = 0.2,
                       epochs = 20,
                       batch_size = 32
                  )

Epoch 1/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - f1_score: 0.6685 - loss: 0.4013 - val_f1_score: 0.6678 - val_loss: 0.3381
Epoch 2/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - f1_score: 0.6685 - loss: 0.3329 - val_f1_score: 0.6678 - val_loss: 0.3280
Epoch 3/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - f1_score: 0.6685 - loss: 0.3263 - val_f1_score: 0.6678 - val_loss: 0.3291
Epoch 4/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.6685 - loss: 0.3245 - val_f1_score: 0.6678 - val_loss: 0.3249
Epoch 5/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - f1_score: 0.6685 - loss: 0.3215 - val_f1_score: 0.6678 - val_loss: 0.3213
Epoch 6/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - f1_score: 0.6685 - loss: 0.3199 - val_f1_score: 0.6678 - val_loss: 0.3207
Epoch 7/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - f1_score: 0.6685 - loss: 0.3177 - val_f1_score: 0.6678 - val_loss: 0.3247
Epoch 8/20
992/992 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - f1_score: 0.6685 - loss: 0.3160 - val_f1_score: 0.

In [27]:
# Evaluate the Model
loss, arc1_F1_Score = model_1.evaluate(X_test, y_test)
print('Test Loss :',loss)
print('Test F1_Score : ', arc1_F1_Score)

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - f1_score: 0.6683 - loss: 0.3235
Test Loss : 0.3235304653644562
Test F1_Score :  [0.6683228]


#**Acrhitecture 2**
- Input layer
- Hidden layer 1: 32 neurons
- Hidden layer 2: 16 neurons
- Output layer

In [28]:
model_2 = Sequential([
                       Dense(32, activation='relu', input_shape=(X_train.shape[1], )),
                       Dense(16, activation='relu'),
                       Dense(1, activation='sigmoid')
                      ])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [29]:
# Compile the model
model_2.compile(
                 optimizer = 'adam',
                 loss = 'binary_crossentropy',
                 metrics = [f1_metric]
)

In [30]:
# Model summary
model_2.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 32)             │         3,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,777 (14.75 KB)

 Trainable params: 3,777 (14.75 KB)

 Non-trainable params: 0 (0.00 B)

In [31]:
# Train the model
history = model_2.fit(X_train, y_train,
                      validation_data = (X_test, y_test),
                      epochs = 20,
                      batch_size = 32
                    )

Epoch 1/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - f1_score: 0.8394 - loss: 0.3666 - val_f1_score: 0.8612 - val_loss: 0.3324
Epoch 2/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - f1_score: 0.8601 - loss: 0.3289 - val_f1_score: 0.8648 - val_loss: 0.3288
Epoch 3/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8628 - loss: 0.3223 - val_f1_score: 0.8632 - val_loss: 0.3253
Epoch 4/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8648 - loss: 0.3179 - val_f1_score: 0.8599 - val_loss: 0.3281
Epoch 5/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8662 - loss: 0.3154 - val_f1_score: 0.8655 - val_loss: 0.3220
Epoch 6/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - f1_score: 0.8678 - loss: 0.3114 - val_f1_score: 0.8568 - val_loss: 0.3250
Epoch 7/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8691 - loss: 0.3083 - val_f1_score: 0.8597 - val_loss: 0.3228
Epoch 8/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8696 - loss: 0.3059 - 

In [32]:
# Evaluate the model
loss, arch2_F1_Score = model_2.evaluate(X_test, y_test)
print('Test Loss : ',loss)
print('Test F1_Score',arch2_F1_Score)

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - f1_score: 0.8656 - loss: 0.3400
Test Loss :  0.340003103017807
Test F1_Score [0.8656459]


#**Architecture 3 :**

- Input layer
- Hidden layer 1: 128 neurons
- Hidden layer 2: 64 neurons
- Hidden layer 3: 32 neurons
- Hidden layer 4: 16 neurons
- Output layer

In [33]:
# Deep Network
model_3 = Sequential([
                    Dense(128,activation='relu', input_shape= (X_train.shape[1],)),
                    Dense(64, activation = 'relu'),
                    Dense(32, activation = 'relu'),
                    Dense(16, activation = 'relu'),
                    Dense(1, activation='sigmoid')
                  ])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [34]:
# Compile
model_3.compile(
                 optimizer = 'adam',
                 loss = 'binary_crossentropy',
                 metrics = [f1_metric]
)

In [35]:
# Model summary
model_3.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_5 (Dense)                 │ (None, 128)            │        12,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,809 (93.00 KB)

 Trainable params: 23,809 (93.00 KB)

 Non-trainable params: 0 (0.00 B)

In [36]:
# Train the model
history = model_3.fit(
                        X_train,y_train,
                        validation_data = (X_test, y_test),
                        epochs = 20,
                        batch_size = 32
                      )

Epoch 1/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - f1_score: 0.8520 - loss: 0.3508 - val_f1_score: 0.8596 - val_loss: 0.3272
Epoch 2/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - f1_score: 0.8610 - loss: 0.3261 - val_f1_score: 0.8137 - val_loss: 0.3684
Epoch 3/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8660 - loss: 0.3177 - val_f1_score: 0.8681 - val_loss: 0.3206
Epoch 4/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - f1_score: 0.8679 - loss: 0.3121 - val_f1_score: 0.8677 - val_loss: 0.3316
Epoch 5/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8716 - loss: 0.3060 - val_f1_score: 0.8690 - val_loss: 0.3315
Epoch 6/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8739 - loss: 0.3004 - val_f1_score: 0.8638 - val_loss: 0.3193
Epoch 7/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - f1_score: 0.8768 - loss: 0.2947 - val_f1_score: 0.8634 - val_loss: 0.3200
Epoch 8/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8770 - loss: 0.2904 -

In [37]:
# Evaluate the model
loss, arch3_F1_Score = model_3.evaluate(X_test, y_test)
print('Test Loss : ', loss)
print('Test F1 Score :',arch3_F1_Score)

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - f1_score: 0.8494 - loss: 0.3975
Test Loss :  0.39746394753456116
Test F1 Score : [0.84938884]


#**Architecture 4 :**

- Input layer
- Hidden layer 1: 256 neurons
- Hidden layer 2: 128 neurons
- Hidden layer 3: 64 neurons
- Hidden layer 4: 1 neurons
- Output layer

In [38]:
# Wide Network
model_4 = Sequential([
                       Dense(256, activation = 'relu', input_shape = (X_train.shape[1],)),
                       Dense(128, activation= 'relu'),
                       Dense(64, activation = 'relu'),
                       Dense(1, activation = 'sigmoid')
                    ])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [39]:
# compile
model_4.compile(
                optimizer ='adam',
                loss = 'binary_crossentropy',
                metrics = [f1_metric])

In [40]:
# Model summary
model_4.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                │ (None, 256)            │        25,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67,073 (262.00 KB)

 Trainable params: 67,073 (262.00 KB)

 Non-trainable params: 0 (0.00 B)

In [41]:
# Train the model
history = model_4.fit(
                      X_train, y_train,
                      validation_data = (X_test, y_test),
                      epochs = 20,
                      batch_size = 32
                     )

Epoch 1/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - f1_score: 0.8486 - loss: 0.3469 - val_f1_score: 0.8678 - val_loss: 0.3328
Epoch 2/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - f1_score: 0.8620 - loss: 0.3228 - val_f1_score: 0.8607 - val_loss: 0.3213
Epoch 3/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8646 - loss: 0.3169 - val_f1_score: 0.8605 - val_loss: 0.3200
Epoch 4/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8692 - loss: 0.3095 - val_f1_score: 0.8654 - val_loss: 0.3256
Epoch 5/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - f1_score: 0.8717 - loss: 0.3040 - val_f1_score: 0.8621 - val_loss: 0.3209
Epoch 6/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - f1_score: 0.8754 - loss: 0.2965 - val_f1_score: 0.8695 - val_loss: 0.3295
Epoch 7/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8791 - loss: 0.2894 - val_f1_score: 0.8628 - val_loss: 0.3306
Epoch 8/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8826 - loss: 0.2807 - 

In [42]:
# Evaluate the model
loss, arc4_F1_Score = model_4.evaluate(
                                        X_test, y_test
                                      )
print('Test Loss : ', loss)
print('Test F1 Score :',arc4_F1_Score)

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - f1_score: 0.8596 - loss: 0.4488
Test Loss :  0.4488442540168762
Test F1 Score : [0.8596319]


#**Architecture 5 :**

- Input layer
- Hidden layer 1: 128 neurons
- BatchNormalization
- Dropout
- Hidden layer 2: 64 neurons
- Hidden layer 3: 32 neurons
- Hidden layer 4: 1 neurons
- Output layer

In [43]:
model_5 = Sequential([
                       Dense(128, activation= 'relu', input_shape = (X_train.shape[1],)),
                       BatchNormalization(),
                       Dropout(0.3),
                       Dense(64, activation = 'relu'),
                       Dense(32, activation ='relu'),
                       Dense(1, activation= 'sigmoid')
                    ])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [44]:
# compile
model_5.compile(
                optimizer ='adam',
                loss = 'binary_crossentropy',
                metrics = [f1_metric]
               )

In [45]:
# model summary
model_5.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_14 (Dense)                │ (None, 128)            │        12,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,809 (93.00 KB)

 Trainable params: 23,553 (92.00 KB)

 Non-trainable params: 256 (1.00 KB)

In [46]:
# Train the model
history = model_5.fit(
                       X_train, y_train,
                       validation_data = (X_test, y_test),
                       epochs = 20,
                       batch_size = 32
                      )

Epoch 1/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - f1_score: 0.8399 - loss: 0.3727 - val_f1_score: 0.8619 - val_loss: 0.3282
Epoch 2/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8533 - loss: 0.3435 - val_f1_score: 0.8607 - val_loss: 0.3489
Epoch 3/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - f1_score: 0.8558 - loss: 0.3383 - val_f1_score: 0.8676 - val_loss: 0.3308
Epoch 4/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8577 - loss: 0.3341 - val_f1_score: 0.8643 - val_loss: 0.3223
Epoch 5/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8573 - loss: 0.3339 - val_f1_score: 0.8664 - val_loss: 0.3261
Epoch 6/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - f1_score: 0.8582 - loss: 0.3313 - val_f1_score: 0.8646 - val_loss: 0.3295
Epoch 7/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8636 - loss: 0.3255 - val_f1_score: 0.8708 - val_loss: 0.3179
Epoch 8/20
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - f1_score: 0.8617 - loss: 0.3250 -

In [47]:
# Evaluate the model
loss, arch5_F1_Score = model_5.evaluate(X_test, y_test)
print('Test Loss : ', loss)
print('Test F1 Score :',arch5_F1_Score)

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - f1_score: 0.8577 - loss: 0.3272
Test Loss :  0.3271697461605072
Test F1 Score : [0.8576697]


# Architecture 6

- Hyperparameter Tuning

In [48]:
def hyperparameter(trail):
  hidden_units = trail.suggest_int('hidden_units', 64, 512)
  dropout_rate = trail.suggest_float('dropout_rate', 0.1, 0.5)
  learning_rate = trail.suggest_float('learning_rate', 1e-4, 1e-2, log = True)
  optimizer_name = trail.suggest_categorical('optimizer', ['adam', 'sgd'])

  # Model Defnition
  model_6 = keras.Sequential([
                             layers.Input(shape = (X_train.shape[1],)),
                             layers.Dense(hidden_units, activation = 'relu'),
                             layers.Dropout(dropout_rate),
                             layers.Dense(1, activation = 'sigmoid')
                          ])
  if optimizer_name == 'sgd':
    optimizer = getattr(keras.optimizers, optimizer_name.upper())(learning_rate = learning_rate)
  else:
    optimizer = getattr(keras.optimizers, optimizer_name.title())(learning_rate = learning_rate)

  # Model compile
  model_6.compile(
                 optimizer = optimizer,
                 loss = 'binary_crossentropy',
                 metrics = [f1_metric]
               )

  # Train the model
  history = model_6.fit(
                         X_train, y_train,
                         validation_split = 0.2,
                         epochs = 100,
                         batch_size = 128,
                         verbose= 0
                      )

  # Returning the best validation f1 score
  return float(max(float(x[0]) for x in history.history['val_f1_score']))

In [49]:
pip install optuna

In [50]:
import optuna

In [51]:
study = optuna.create_study(direction = 'maximize')
study.optimize(hyperparameter, n_trials = 20)
print('Best Trial:')
trial = study.best_trial
arch6_F1_Score = trial.value
print(f'F1 Score : {arch6_F1_Score}')
for key, value in trial.params.items():
  print(f'{key} : {value}')

[I 2026-11-12 11:39:41,673] A new study created in memory with name: no-name-76eb57c7-f644-4fa4-a746-59f8547ecb81
[I 2026-11-12 11:41:28,560] Trial 0 finished with value: 0.8704754114151001 and parameters: {'hidden_units': 433, 'dropout_rate': 0.32764943476298003, 'learning_rate': 0.0003210976597346885, 'optimizer': 'adam'}. Best is trial 0 with value: 0.8704754114151001.
[I 2026-11-12 11:43:09,312] Trial 1 finished with value: 0.800837516784668 and parameters: {'hidden_units': 359, 'dropout_rate': 0.41789108748670345, 'learning_rate': 0.00015088330055919517, 'optimizer': 'sgd'}. Best is trial 0 with value: 0.8704754114151001.
[I 2026-11-12 11:44:51,474] Trial 2 finished with value: 0.8577154278755188 and parameters: {'hidden_units': 349, 'dropout_rate': 0.13347196092754496, 'learning_rate': 0.0011113192667820966, 'optimizer': 'sgd'}. Best is trial 0 with value: 0.8704754114151001.
[I 2026-11-12 11:46:36,919] Trial 3 finished with value: 0.7968480587005615 and parameters: {'hidden_unit

Best Trial:
F1 Score : 0.8711144328117371
hidden_units : 107
dropout_rate : 0.3115400736743442
learning_rate : 0.003949905222821573
optimizer : adam


In [52]:
best_params = study.best_trial.params
hidden_units = best_params['hidden_units']
dropout_rate = best_params['dropout_rate']
learning_rate = best_params['learning_rate']
optimizer_name = best_params['optimizer']
print('Best Hyperparameters :', best_params)

Best Hyperparameters : {'hidden_units': 107, 'dropout_rate': 0.3115400736743442, 'learning_rate': 0.003949905222821573, 'optimizer': 'adam'}


In [53]:
# Model building
model_6 = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),

    layers.Dense(hidden_units, activation='relu'),
    layers.Dropout(dropout_rate),

    layers.Dense(1, activation='sigmoid')
])

# Select optimizer
if optimizer_name == 'adam':
    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate
    )
else:
    optimizer = keras.optimizers.SGD(
        learning_rate=learning_rate
    )

# Compile the model
model_6.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=[f1_metric]
)

# Display summary
model_6.summary()

Model: "sequential_25"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_58 (Dense)                │ (None, 107)            │        10,807 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 107)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_59 (Dense)                │ (None, 1)              │           108 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,915 (42.64 KB)

 Trainable params: 10,915 (42.64 KB)

 Non-trainable params: 0 (0.00 B)

In [54]:
# Train the model
history = model_6.fit(
                    X_train,
                    y_train,
                    validation_split = 0.2,
                    batch_size=128,
                    epochs=100,
                    verbose=1
                  )

Epoch 1/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - f1_score: 0.8369 - loss: 0.3799 - val_f1_score: 0.8555 - val_loss: 0.3557
Epoch 2/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - f1_score: 0.8550 - loss: 0.3381 - val_f1_score: 0.8634 - val_loss: 0.3241
Epoch 3/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - f1_score: 0.8588 - loss: 0.3321 - val_f1_score: 0.8514 - val_loss: 0.3287
Epoch 4/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - f1_score: 0.8616 - loss: 0.3255 - val_f1_score: 0.8658 - val_loss: 0.3255
Epoch 5/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - f1_score: 0.8620 - loss: 0.3243 - val_f1_score: 0.8600 - val_loss: 0.3210
Epoch 6/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - f1_score: 0.8641 - loss: 0.3226 - val_f1_score: 0.8621 - val_loss: 0.3178
Epoch 7/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - f1_score: 0.8673 - loss: 0.3183 - val_f1_score: 0.8617 - val_loss: 0.3199
Epoch 8/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - f1_score: 0.8675 - loss: 0.3177 - val_f1_

In [55]:
# Evaluate
loss, arch6_F1_Score = model_6.evaluate(
                                          X_test,
                                          y_test,
                                         )

print(f'Test Loss: {loss}')
print(f'Test Accuracy: {arch6_F1_Score}')

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - f1_score: 0.8663 - loss: 0.4055
Test Loss: 0.4055091142654419
Test Accuracy: [0.8662773]


# DataFrame


In [56]:
# Creating all architecture F1 Scores DataFrame
df = pd.DataFrame({
                  'Architecture': [
                              'Architecture 1 (Shallow Network)',
                              'Architecture 2 ',
                              'Architecture 3 (Deep Network)',
                              'Architecture 4 (Wide Network) ',
                              'Architecture 5 (BatchNormalization & Dropout)',
                              'Architecture 6 (Hyperparameter Tuning)'
                              ],

                  'F1_Scores': [
                              arc1_F1_Score,
                              arch2_F1_Score,
                              arch3_F1_Score,
                              arc4_F1_Score,
                              arch5_F1_Score,
                              arch6_F1_Score
                              ]
                  })



In [57]:
df = df.sort_values(by='F1_Scores', ascending= False).reset_index(drop=True)
df['F1_Scores'] = df['F1_Scores']*100
df

,Architecture,F1_Scores
0,Architecture 6 (Hyperparameter Tuning),[86.62773]
1,Architecture 2,[86.56459]
2,Architecture 4 (Wide Network),[85.96319]
3,Architecture 5 (BatchNormalization & Dropout),[85.76697]
4,Architecture 3 (Deep Network),[84.93888]
5,Architecture 1 (Shallow Network),[66.83228]


**Observation:**

Out all 6 architectures iam choosing the 2nd architecture cazz out of all architectures in this architecture parameter is simple and it gives better perfomance.

# Saving the Model

**Save your best model**

In [58]:
import joblib

# Save ANN model
model_2.save("IMDB_Sentiment_Analysis_(Fast_Text&ANN).keras")

# Save FastText model
Fast_text_model.save("FastText_model.model")